# Chapter 3: Calculus for Machine Learning

This notebook covers the calculus concepts essential for understanding how neural networks learn:

1. Numerical Gradients
2. Automatic Differentiation Concept
3. Chain Rule Visualization
4. Gradient Descent on Simple Functions

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact

# Configure matplotlib
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 1. Numerical Gradients

The gradient tells us how a function changes with respect to its inputs. We can approximate it numerically using finite differences:

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + h \cdot e_i) - f(x - h \cdot e_i)}{2h}$$

This is the **central difference** formula, which is more accurate than forward/backward differences.

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    """
    Compute the numerical gradient of f at point x using central differences.
    
    Parameters:
        f: function that takes array x and returns scalar
        x: point at which to compute gradient
        h: step size for finite differences
    
    Returns:
        grad: numerical gradient (same shape as x)
    """
    x = np.array(x, dtype=float)
    grad = np.zeros_like(x)
    
    for i in range(len(x)):
        # Create perturbation vectors
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[i] += h
        x_minus[i] -= h
        
        # Central difference
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    
    return grad

In [ ]:
# Test on a simple quadratic function
def quadratic(x):
    """f(x, y) = x^2 + 2y^2"""
    return x[0]**2 + 2*x[1]**2

def quadratic_analytical_gradient(x):
    """Analytical gradient: [2x, 4y]"""
    return np.array([2*x[0], 4*x[1]])

# Compare numerical and analytical gradients
test_points = [
    np.array([1.0, 1.0]),
    np.array([3.0, -2.0]),
    np.array([0.0, 0.0]),
    np.array([-1.5, 2.5])
]

print("Comparing Numerical vs Analytical Gradients:")
print("-" * 60)
for x in test_points:
    num_grad = numerical_gradient(quadratic, x)
    ana_grad = quadratic_analytical_gradient(x)
    error = np.linalg.norm(num_grad - ana_grad)
    
    print(f"Point: {x}")
    print(f"  Numerical:  {num_grad}")
    print(f"  Analytical: {ana_grad}")
    print(f"  Error: {error:.2e}\n")

In [ ]:
# Analyze the effect of step size h
def f_simple(x):
    return np.sin(x[0]) * np.exp(-x[1]**2)

def f_simple_analytical_grad(x):
    return np.array([
        np.cos(x[0]) * np.exp(-x[1]**2),
        -2 * x[1] * np.sin(x[0]) * np.exp(-x[1]**2)
    ])

x_test = np.array([1.0, 0.5])
true_grad = f_simple_analytical_grad(x_test)

h_values = np.logspace(-12, 0, 50)
errors = []

for h in h_values:
    num_grad = numerical_gradient(f_simple, x_test, h=h)
    errors.append(np.linalg.norm(num_grad - true_grad))

plt.figure(figsize=(10, 6))
plt.loglog(h_values, errors, 'b.-', linewidth=2, markersize=4)
plt.xlabel('Step size h', fontsize=12)
plt.ylabel('Gradient Error', fontsize=12)
plt.title('Numerical Gradient Error vs Step Size', fontsize=14)
plt.grid(True, alpha=0.3)

# Mark optimal region
optimal_h = h_values[np.argmin(errors)]
plt.axvline(x=optimal_h, color='r', linestyle='--', 
            label=f'Optimal h = {optimal_h:.2e}')
plt.legend()
plt.show()

print(f"Optimal step size: h = {optimal_h:.2e}")
print("\nToo small: numerical precision issues (floating point errors)")
print("Too large: approximation errors (truncation errors)")

## 2. Automatic Differentiation Concept

Automatic differentiation (autodiff) computes exact derivatives by tracking operations. Modern ML frameworks like PyTorch use this.

Let's build a simple autodiff system to understand the concept:

In [ ]:
class Value:
    """
    A simple autodiff class that tracks values and gradients.
    Inspired by Andrej Karpathy's micrograd.
    """
    
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            # d(a+b)/da = 1, d(a+b)/db = 1
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            # d(a*b)/da = b, d(a*b)/db = a
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')
        
        def _backward():
            # d(x^n)/dx = n * x^(n-1)
            self.grad += n * (self.data ** (n-1)) * out.grad
        out._backward = _backward
        return out
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __radd__(self, other):
        return self + other
    
    def __rmul__(self, other):
        return self * other
    
    def backward(self):
        """Compute gradients via reverse-mode autodiff (backpropagation)."""
        # Topological sort
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        # Backward pass
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [ ]:
# Example: f(x, y) = x^2 + 2*x*y + y^2 = (x + y)^2
x = Value(3.0)
y = Value(4.0)

# Build computation graph
f = x**2 + 2*x*y + y**2

print(f"f(3, 4) = {f.data}")
print(f"Expected: (3+4)^2 = {(3+4)**2}")

# Compute gradients
f.backward()

print(f"\ndf/dx at (3, 4): {x.grad}")
print(f"Expected: 2x + 2y = 2*3 + 2*4 = {2*3 + 2*4}")

print(f"\ndf/dy at (3, 4): {y.grad}")
print(f"Expected: 2x + 2y = 2*3 + 2*4 = {2*3 + 2*4}")

In [ ]:
# A simple neuron computation
# z = w1*x1 + w2*x2 + b
# output = z^2 (simplified activation)

# Inputs
x1 = Value(2.0)
x2 = Value(3.0)

# Weights and bias
w1 = Value(0.5)
w2 = Value(-0.3)
b = Value(0.1)

# Forward pass
z = w1*x1 + w2*x2 + b
output = z ** 2

print("Forward pass:")
print(f"  z = w1*x1 + w2*x2 + b = {z.data:.4f}")
print(f"  output = z^2 = {output.data:.4f}")

# Backward pass
output.backward()

print("\nGradients (for gradient descent):")
print(f"  d(output)/dw1 = {w1.grad:.4f}")
print(f"  d(output)/dw2 = {w2.grad:.4f}")
print(f"  d(output)/db  = {b.grad:.4f}")

## 3. Chain Rule Visualization

The chain rule is the foundation of backpropagation:

$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial x}$$

For a composition $f(g(h(x)))$:

$$\frac{df}{dx} = \frac{df}{dg} \cdot \frac{dg}{dh} \cdot \frac{dh}{dx}$$

In [ ]:
# Visualize the chain rule with a composition of functions
# f(x) = sin(x^2)

def h(x):
    """Inner function: h(x) = x^2"""
    return x**2

def g(u):
    """Outer function: g(u) = sin(u)"""
    return np.sin(u)

def f(x):
    """Composition: f(x) = g(h(x)) = sin(x^2)"""
    return g(h(x))

# Derivatives
def dh_dx(x):
    """dh/dx = 2x"""
    return 2*x

def dg_du(u):
    """dg/du = cos(u)"""
    return np.cos(u)

def df_dx(x):
    """Chain rule: df/dx = dg/du * du/dx = cos(x^2) * 2x"""
    return dg_du(h(x)) * dh_dx(x)

# Visualize
x = np.linspace(-2, 2, 200)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot h(x) = x^2
ax = axes[0, 0]
ax.plot(x, h(x), 'b-', linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('h(x)')
ax.set_title(r'$h(x) = x^2$ (inner function)')
ax.grid(True, alpha=0.3)

# Plot g(u) = sin(u)
ax = axes[0, 1]
u = np.linspace(0, 4, 200)
ax.plot(u, g(u), 'r-', linewidth=2)
ax.set_xlabel('u')
ax.set_ylabel('g(u)')
ax.set_title(r'$g(u) = \sin(u)$ (outer function)')
ax.grid(True, alpha=0.3)

# Plot f(x) = sin(x^2)
ax = axes[1, 0]
ax.plot(x, f(x), 'g-', linewidth=2)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title(r'$f(x) = \sin(x^2)$ (composition)')
ax.grid(True, alpha=0.3)

# Plot derivative via chain rule
ax = axes[1, 1]
ax.plot(x, df_dx(x), 'purple', linewidth=2, label=r"$f'(x) = 2x\cos(x^2)$")
ax.plot(x, dh_dx(x), 'b--', linewidth=1, alpha=0.7, label=r"$h'(x) = 2x$")
ax.plot(x, dg_du(h(x)), 'r--', linewidth=1, alpha=0.7, label=r"$g'(h(x)) = \cos(x^2)$")
ax.set_xlabel('x')
ax.set_ylabel('Derivative')
ax.set_title('Chain Rule: $f\'(x) = g\'(h(x)) \cdot h\'(x)$')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize backpropagation through a simple network
def visualize_backprop():
    """
    Visualize forward and backward passes through a computation graph.
    
    Computation: L = (x * w + b)^2
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Values
    x_val, w_val, b_val = 2.0, 3.0, 1.0
    z_val = x_val * w_val + b_val  # = 7
    L_val = z_val ** 2  # = 49
    
    # Forward pass visualization
    ax = axes[0]
    ax.set_xlim(-1, 6)
    ax.set_ylim(-0.5, 3)
    
    # Nodes
    nodes = {
        'x': (0, 2), 'w': (0, 1), 'b': (0, 0),
        'x*w': (2, 1.5), 'z': (4, 1), 'L': (5.5, 1)
    }
    
    values = {
        'x': x_val, 'w': w_val, 'b': b_val,
        'x*w': x_val*w_val, 'z': z_val, 'L': L_val
    }
    
    for name, (px, py) in nodes.items():
        circle = plt.Circle((px, py), 0.25, color='lightblue', ec='blue', linewidth=2)
        ax.add_patch(circle)
        ax.text(px, py, f"{name}\n{values[name]}", ha='center', va='center', fontsize=9)
    
    # Edges (forward)
    edges = [
        ('x', 'x*w'), ('w', 'x*w'), ('x*w', 'z'), ('b', 'z'), ('z', 'L')
    ]
    for start, end in edges:
        sx, sy = nodes[start]
        ex, ey = nodes[end]
        ax.annotate('', xy=(ex-0.25, ey), xytext=(sx+0.25, sy),
                   arrowprops=dict(arrowstyle='->', color='green', lw=2))
    
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Forward Pass', fontsize=14)
    
    # Backward pass visualization
    ax = axes[1]
    ax.set_xlim(-1, 6)
    ax.set_ylim(-0.5, 3)
    
    # Gradients (computed via chain rule)
    dL_dL = 1
    dL_dz = 2 * z_val  # d(z^2)/dz = 2z = 14
    dL_db = dL_dz * 1  # = 14
    dL_dxw = dL_dz * 1  # = 14
    dL_dx = dL_dxw * w_val  # = 14 * 3 = 42
    dL_dw = dL_dxw * x_val  # = 14 * 2 = 28
    
    grads = {
        'x': dL_dx, 'w': dL_dw, 'b': dL_db,
        'x*w': dL_dxw, 'z': dL_dz, 'L': dL_dL
    }
    
    for name, (px, py) in nodes.items():
        circle = plt.Circle((px, py), 0.25, color='lightyellow', ec='orange', linewidth=2)
        ax.add_patch(circle)
        ax.text(px, py, f"{name}\ngrad={grads[name]}", ha='center', va='center', fontsize=8)
    
    # Edges (backward - reversed)
    for end, start in edges:
        sx, sy = nodes[start]
        ex, ey = nodes[end]
        ax.annotate('', xy=(ex+0.25, ey), xytext=(sx-0.25, sy),
                   arrowprops=dict(arrowstyle='->', color='red', lw=2))
    
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Backward Pass (Gradients)', fontsize=14)
    
    plt.tight_layout()
    plt.show()

visualize_backprop()

## 4. Gradient Descent on Simple Functions

Gradient descent finds function minima by following the negative gradient:

$$x_{t+1} = x_t - \eta \nabla f(x_t)$$

where $\eta$ is the learning rate.

In [ ]:
# 1D Gradient Descent
def f_1d(x):
    """A function with multiple local minima."""
    return x**4 - 3*x**3 + 2*x**2 + x - 1

def df_1d(x):
    """Derivative of f_1d."""
    return 4*x**3 - 9*x**2 + 4*x + 1

def gradient_descent_1d(f, df, x0, lr=0.01, n_steps=100):
    """Run gradient descent and return the path."""
    path = [x0]
    x = x0
    for _ in range(n_steps):
        x = x - lr * df(x)
        path.append(x)
    return np.array(path)

# Visualize
x = np.linspace(-1, 3, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different starting points
ax = axes[0]
ax.plot(x, f_1d(x), 'b-', linewidth=2, label='f(x)')

for x0, color in [(2.5, 'red'), (0.5, 'green'), (-0.5, 'orange')]:
    path = gradient_descent_1d(f_1d, df_1d, x0, lr=0.01, n_steps=100)
    ax.plot(path, f_1d(path), 'o-', color=color, markersize=3, 
            alpha=0.7, label=f'Start={x0}')
    ax.scatter(path[0], f_1d(path[0]), color=color, s=100, zorder=5)

ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Gradient Descent from Different Starting Points')
ax.legend()
ax.grid(True, alpha=0.3)

# Different learning rates
ax = axes[1]
ax.plot(x, f_1d(x), 'b-', linewidth=2, label='f(x)')

x0 = 2.0
for lr, color in [(0.001, 'red'), (0.01, 'green'), (0.05, 'orange')]:
    path = gradient_descent_1d(f_1d, df_1d, x0, lr=lr, n_steps=50)
    ax.plot(path, f_1d(path), 'o-', color=color, markersize=3,
            alpha=0.7, label=f'lr={lr}')

ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Effect of Learning Rate')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 2D Gradient Descent
def f_2d(x):
    """Rosenbrock function - a classic optimization benchmark."""
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

def grad_f_2d(x):
    """Gradient of Rosenbrock function."""
    dfdx0 = -2*(1 - x[0]) - 400*x[0]*(x[1] - x[0]**2)
    dfdx1 = 200*(x[1] - x[0]**2)
    return np.array([dfdx0, dfdx1])

def gradient_descent_2d(grad_f, x0, lr=0.001, n_steps=1000):
    """Run gradient descent in 2D."""
    path = [x0.copy()]
    x = x0.copy()
    for _ in range(n_steps):
        x = x - lr * grad_f(x)
        path.append(x.copy())
    return np.array(path)

# Create contour plot
x_range = np.linspace(-2, 2, 100)
y_range = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = f_2d(np.array([X[i, j], Y[i, j]]))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Contour plot with path
ax = axes[0]
levels = np.logspace(0, 3, 20)
ax.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=0.7)
ax.contourf(X, Y, Z, levels=levels, cmap='viridis', alpha=0.3)

# Run gradient descent
x0 = np.array([-1.5, 2.0])
path = gradient_descent_2d(grad_f_2d, x0, lr=0.001, n_steps=5000)

ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=2, linewidth=1, alpha=0.8)
ax.scatter(path[0, 0], path[0, 1], color='red', s=100, zorder=5, label='Start')
ax.scatter(1, 1, color='green', s=100, marker='*', zorder=5, label='Minimum (1,1)')

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Gradient Descent on Rosenbrock Function')
ax.legend()

# Convergence plot
ax = axes[1]
losses = [f_2d(p) for p in path]
ax.semilogy(losses, 'b-', linewidth=1)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Convergence')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Starting point: {x0}")
print(f"Final point: {path[-1]}")
print(f"True minimum: (1, 1)")
print(f"Final loss: {f_2d(path[-1]):.6f}")

In [ ]:
# Interactive gradient descent visualization
def interactive_gd(lr=0.01, n_steps=50):
    """Interactive gradient descent on a quadratic."""
    def f(x):
        return x[0]**2 + 4*x[1]**2  # Elliptical bowl
    
    def grad_f(x):
        return np.array([2*x[0], 8*x[1]])
    
    x0 = np.array([4.0, 3.0])
    path = gradient_descent_2d(grad_f, x0, lr=lr, n_steps=n_steps)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Contour plot
    ax = axes[0]
    x_range = np.linspace(-5, 5, 100)
    y_range = np.linspace(-4, 4, 100)
    X, Y = np.meshgrid(x_range, y_range)
    Z = X**2 + 4*Y**2
    
    ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
    ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=5, linewidth=2)
    ax.scatter(path[0, 0], path[0, 1], color='red', s=100, zorder=5)
    ax.scatter(0, 0, color='green', s=100, marker='*', zorder=5)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f'lr={lr}, steps={n_steps}')
    ax.set_aspect('equal')
    
    # Loss curve
    ax = axes[1]
    losses = [f(p) for p in path]
    ax.semilogy(losses, 'b-', linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('Convergence')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Final loss: {losses[-1]:.6f}")

interact(interactive_gd,
         lr=widgets.FloatSlider(min=0.001, max=0.3, step=0.01, value=0.1, description='Learning rate'),
         n_steps=widgets.IntSlider(min=10, max=200, step=10, value=50, description='Steps'));

## Summary

In this notebook, we covered:

1. **Numerical Gradients**: Approximating derivatives using finite differences
2. **Automatic Differentiation**: How modern ML frameworks compute exact gradients
3. **Chain Rule**: The mathematical foundation of backpropagation
4. **Gradient Descent**: Finding minima by following negative gradients

### Key Takeaways:

- Numerical gradients are useful for debugging but slow for training
- Autodiff computes exact gradients efficiently via the chain rule
- Learning rate is critical: too small = slow, too large = divergence
- The loss landscape determines optimization difficulty

---

*Continue to Chapter 6 for SVD applications...*